In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv("Output/netflix_cleaned.csv")
print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()

Dataset loaded successfully!
Rows: 8790
Columns: 10


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [4]:
features = [
    "type",
    "director",
    "country",
    "rating",
    "listed_in"
]

for col in features:
    df[col] = df[col].fillna("Unknown").astype(str)

df["combined_features"] = df[features].agg(
    " ".join,
    axis=1
)
display(df[["title", "combined_features"]].head())

,title,combined_features
0,Dick Johnson Is Dead,Movie Kirsten Johnson United States PG-13 Docu...
1,Ganglands,TV Show Julien Leclercq France TV-MA Crime TV ...
2,Midnight Mass,TV Show Mike Flanagan United States TV-MA TV D...
3,Confessions of an Invisible Girl,Movie Bruno Garotti Brazil TV-PG Children & Fa...
4,Sankofa,"Movie Haile Gerima United States TV-MA Dramas,..."


In [5]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(
    df["combined_features"]
)
print("TF-IDF conversion completed!")
print("Matrix shape:", tfidf_matrix.shape)

TF-IDF conversion completed!
Matrix shape: (8790, 6569)


In [6]:
similarity_matrix = cosine_similarity(
    tfidf_matrix,
    dense_output=False
)
print("Similarity calculation completed!")

Similarity calculation completed!


In [7]:
def recommend_titles(title, top_n=5):
    matches = df.index[
        df["title"].str.casefold() == title.casefold()
    ].tolist()

    if not matches:
        print("Title not found in the dataset.")
        return
    idx = matches[0]
    scores = similarity_matrix[idx].toarray().ravel()

    # Exclude the selected title itself
    scores[idx] = -1

    similar_indices = scores.argsort()[::-1][:top_n]

    recommendations = df.iloc[similar_indices][
        ["title", "type", "listed_in", "release_year"]
    ].copy()

    recommendations["similarity_score"] = scores[similar_indices]
    return recommendations

In [8]:
recommend_titles("Ganglands", 5)

,title,type,listed_in,release_year,similarity_score
1183,Sentinelle,Movie,"Action & Adventure, Dramas, International Movies",2021,0.847388
2142,Earth and Blood,Movie,"Dramas, International Movies, Thrillers",2020,0.773519
8408,Crime Time,TV Show,"Crime TV Shows, International TV Shows, TV Act...",2017,0.684723
6738,Lupin,TV Show,"Crime TV Shows, International TV Shows, TV Act...",2021,0.684723
8065,La Mante,TV Show,"Crime TV Shows, International TV Shows, TV Dramas",2017,0.616748


In [9]:
df["title"].head(20).tolist()

['Dick Johnson Is Dead',
 'Ganglands',
 'Midnight Mass',
 'Confessions of an Invisible Girl',
 'Sankofa',
 'The Great British Baking Show',
 'The Starling',
 'Motu Patlu in the Game of Zones',
 'Je Suis Karl',
 'Motu Patlu in Wonderland',
 'Motu Patlu: Deep Sea Adventure',
 'Motu Patlu: Mission Moon',
 '99 Songs (Tamil)',
 'Bridgerton - The Afterparty',
 'Bling Empire - The Afterparty',
 'Cobra Kai - The Afterparty',
 'The Circle - The Afterparty',
 'Jailbirds New Orleans',
 'Crime Stories: India Detectives',
 'True: Winter Wishes']

In [10]:
recommend_titles("The Crown", 5)

,title,type,listed_in,release_year,similarity_score
7799,Black Earth Rising,TV Show,"British TV Shows, International TV Shows, TV D...",2018,1.0
7745,Traitors,TV Show,"British TV Shows, International TV Shows, TV D...",2019,1.0
8026,Requiem,TV Show,"British TV Shows, International TV Shows, TV D...",2018,1.0
7677,Black Mirror,TV Show,"British TV Shows, International TV Shows, TV D...",2019,1.0
6867,Behind Her Eyes,TV Show,"British TV Shows, International TV Shows, TV D...",2021,1.0


In [11]:
recommend_titles("Breaking Bad", 5)

,title,type,listed_in,release_year,similarity_score
7674,Designated Survivor,TV Show,"Crime TV Shows, TV Dramas, TV Thrillers",2019,1.000000
7309,Ozark,TV Show,"Crime TV Shows, TV Dramas, TV Thrillers",2020,1.000000
6724,The Assassination of Gianni Versace,TV Show,"Crime TV Shows, TV Dramas, TV Thrillers",2018,1.000000
6925,Dare Me,TV Show,"Crime TV Shows, TV Dramas, TV Thrillers",2019,1.000000
7078,The Blacklist,TV Show,"Crime TV Shows, TV Dramas, TV Thrillers",2019,0.947209


# Recommendation System — Conclusion

## Objective
To develop a content-based recommendation system for Netflix titles.

## Methodology
1. Selected content features such as type, director, country, rating, and genre.
2. Combined the features into a single text representation.
3. Used TF-IDF to convert text into numerical vectors.
4. Calculated cosine similarity between titles.
5. Generated recommendations for a selected title.

## Conclusion
The recommendation system identifies Netflix titles with similar metadata.
It demonstrates how text processing and similarity measures can be used
to build a basic content-based recommendation system.